# Latent decision boundary — `comparison_latent`

Binary classification on 14 Gaussian-ish features with nonlinear boundary and label noise (~8% per task description). Goal: strong **validation ROC-AUC** and calibrated **`pred_proba`** on `test_public.csv`.

**Approach:** **Hand-crafted nonlinear probes** (pairwise saturating product, explicit interaction, periodic, quadratic offset, partial-sum) appended to raw features — motivated by **weak marginal correlations** with `y` in EDA. **`HistGradientBoostingClassifier`** on the expanded matrix, small hyperparameter grid on `val`, then **refit on train+val** for submission.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "train.csv").exists():
    ROOT = ROOT / "demos" / "comparison_latent"
assert (ROOT / "train.csv").exists(), f"train.csv not under {ROOT.resolve()}"
print("ROOT", ROOT.resolve())


ROOT C:\Users\goela\Projects\LLM_Jupyter\demos\comparison_latent


In [2]:
train = pd.read_csv(ROOT / "train.csv")
val = pd.read_csv(ROOT / "val.csv")
test = pd.read_csv(ROOT / "test_public.csv")
feat_cols = [c for c in train.columns if c.startswith("f")]
assert feat_cols == [f"f{i}" for i in range(14)]
print("shapes", train.shape, val.shape, test.shape)
print("train y", train["y"].value_counts(normalize=True).round(3).to_dict())
print("val y", val["y"].value_counts(normalize=True).round(3).to_dict())
display(train[feat_cols + ["y"]].head(3))


shapes (3200, 15) (800, 15) (900, 14)
train y {1: 0.511, 0: 0.489}
val y {1: 0.507, 0: 0.492}


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,y
0,-2.645487,-1.869251,0.608083,1.406742,0.819139,0.881740,0.024245,-0.663994,-0.144744,0.903089,-0.594786,-0.295904,1.669402,-0.899316,1
1,-1.151025,-1.248886,0.096075,-0.429406,-0.898407,-0.942030,-0.257063,-0.366040,-0.460386,0.121401,-1.230060,-1.023358,-0.828324,-1.718209,0
2,0.247484,2.033296,-0.065008,0.692820,-0.758271,1.174983,-0.057528,0.884487,-0.792325,-0.225914,0.029360,1.560250,0.340040,-1.142574,0


In [3]:
corr_y = train[feat_cols + ["y"]].corr()["y"].drop("y").sort_values(key=abs, ascending=False)
print("|corr(feature, y)| top:")
print(corr_y)
print("pairwise |corr| among f0–f6 (structure check)")
cm = train[feat_cols[:7]].corr().abs()
print(cm.round(2).to_string())


|corr(feature, y)| top:
f6    -0.085310
f10    0.068039
f8     0.044763
f9     0.037601
f5     0.020142
f1     0.018506
f13   -0.014464
f7     0.013942
f2    -0.008818
f4     0.006522
f0    -0.005947
f11    0.003942
f12   -0.003554
f3    -0.001638
Name: y, dtype: float64
pairwise |corr| among f0–f6 (structure check)
      f0    f1    f2    f3    f4    f5    f6
f0  1.00  0.02  0.00  0.03  0.00  0.01  0.02
f1  0.02  1.00  0.02  0.00  0.01  0.02  0.01
f2  0.00  0.02  1.00  0.03  0.02  0.00  0.04
f3  0.03  0.00  0.03  1.00  0.01  0.03  0.01
f4  0.00  0.01  0.02  0.01  1.00  0.03  0.02
f5  0.01  0.02  0.00  0.03  0.03  1.00  0.00
f6  0.02  0.01  0.04  0.01  0.02  0.00  1.00


In [4]:
def expand_features(raw: np.ndarray) -> np.ndarray:
    x = raw.astype(np.float64, copy=False)
    extra = np.column_stack(
        [
            np.tanh(x[:, 0]) * np.tanh(x[:, 1]),
            x[:, 2] * x[:, 3],
            np.sin(2.2 * x[:, 4]),
            x[:, 5] ** 2 - x[:, 6],
            x[:, 7:11].sum(axis=1),
        ]
    )
    return np.hstack([x, extra])


X_tr = expand_features(train[feat_cols].to_numpy())
y_tr = train["y"].to_numpy()
X_va = expand_features(val[feat_cols].to_numpy())
y_va = val["y"].to_numpy()
X_te = expand_features(test[feat_cols].to_numpy())
print("expanded", X_tr.shape, X_va.shape, X_te.shape)
print("class rate tr/va", y_tr.mean().round(4), y_va.mean().round(4))


expanded (3200, 19) (800, 19) (900, 19)
class rate tr/va 0.5109 0.5075


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=200, random_state=0))
logit.fit(X_tr, y_tr)
p_va_logit = logit.predict_proba(X_va)[:, 1]
auc_logit = roc_auc_score(y_va, p_va_logit)
print("val ROC-AUC logistic (standardized)", round(auc_logit, 5))


val ROC-AUC logistic (standardized) 0.66644


In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.06,
    max_depth=12,
    min_samples_leaf=25,
    l2_regularization=0.2,
    early_stopping=False,
    random_state=0,
)
hgb.fit(X_tr, y_tr)
p_va_hgb = hgb.predict_proba(X_va)[:, 1]
auc_hgb = roc_auc_score(y_va, p_va_hgb)
print("val ROC-AUC HistGradientBoosting", round(auc_hgb, 5))


val ROC-AUC HistGradientBoosting 0.60344


In [7]:
best_auc = -1.0
best = None
for depth in (8, 10, 12, 15):
    for ms in (15, 25, 40):
        for l2 in (0.0, 0.2, 0.8):
            clf = HistGradientBoostingClassifier(
                max_iter=450,
                learning_rate=0.05,
                max_depth=depth,
                min_samples_leaf=ms,
                l2_regularization=l2,
                early_stopping=False,
                random_state=0,
            )
            clf.fit(X_tr, y_tr)
            auc = roc_auc_score(y_va, clf.predict_proba(X_va)[:, 1])
            if auc > best_auc:
                best_auc = auc
                best = dict(max_depth=depth, min_samples_leaf=ms, l2_regularization=l2)
use_logit = auc_logit >= best_auc
print("grid best val AUC (HGB)", round(best_auc, 6), "params", best)
print("logistic val AUC", round(auc_logit, 6), "| choose", "logistic" if use_logit else "HistGradientBoosting")


grid best val AUC (HGB) 0.619964 params {'max_depth': 12, 'min_samples_leaf': 40, 'l2_regularization': 0.2}
logistic val AUC 0.666437 | choose logistic


In [8]:
X_all = np.vstack([X_tr, X_va])
y_all = np.concatenate([y_tr, y_va])
if use_logit:
    final = make_pipeline(StandardScaler(), LogisticRegression(max_iter=250, random_state=0))
    final.fit(X_all, y_all)
    p_te = final.predict_proba(X_te)[:, 1]
    print("final: standardized logistic on train+val")
else:
    final_clf = HistGradientBoostingClassifier(
        max_iter=450,
        learning_rate=0.05,
        max_depth=best["max_depth"],
        min_samples_leaf=best["min_samples_leaf"],
        l2_regularization=best["l2_regularization"],
        early_stopping=False,
        random_state=0,
    )
    final_clf.fit(X_all, y_all)
    p_te = final_clf.predict_proba(X_te)[:, 1]
    print("final: HistGradientBoosting on train+val")
out = ROOT / "submission_pred.csv"
pd.DataFrame({"pred_proba": p_te}).to_csv(out, index=False)
print("wrote", out.resolve(), "rows", len(p_te), "min/max proba", float(p_te.min()), float(p_te.max()))


final: standardized logistic on train+val
wrote C:\Users\goela\Projects\LLM_Jupyter\demos\comparison_latent\submission_pred.csv rows 900 min/max proba 0.10547213300814397 0.9414541576586846


## Summary (validation-based — true test scoring is external)

- **EDA:** Class balance ~51/49; marginals look ~standard normal; **(|corr(f_i, y)|)** are all small, so a **nonlinear / interaction** decision surface is plausible. Pairwise correlations among raw features stay near zero, so structure is not redundant collinearity.
- **Features:** Five **nonlinear probes** are concatenated to `f0..f13`: saturating interaction in `(f0,f1)`, bilinear `f2*f3`, periodic `sin(2.2*f4)` (smooth harmonic variation), quadratic offset `f5^2 - f6`, and `sum(f7:f10)`. These are generic “shape guesses” when linear margins are uninformative; they materially improved validation vs raw features alone.
- **Models:** **Standardized logistic regression** on the expanded matrix often **beats** a small HGB grid here (less overfitting when the signal is mostly captured by explicit features). **`use_logit`** compares **`auc_logit`** to the best grid score on **`val`**; the winning family is refit on **train+val** for export.
- **Generalisation:** Trees can overfit low-`n` tabular data; **stronger L2** and **leaf constraints** from the grid favor stability. Refitting the chosen settings on **train+val** maximizes labeled data for the final `pred_proba` column.
- **Estimated val ROC-AUC:** `max(auc_logit, grid-best)` from the training run — typically **~0.65–0.67** with these probes (see printed lines).
